In [19]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display

pd.options.display.max_columns = None

In [20]:
lahmans = sql.connect(user='root', password='wassup01L', db='lahmansbaseballdb')

pitching_query = '''
    WITH data AS (
        SELECT *
        FROM pitching
        WHERE yearID > 1947 
            AND ERA < 4.00
            AND (IPouts / 3) > 175
    ), names AS (
        SELECT 
            playerID as trash,
            nameFirst,
            nameLast
        FROM people
    )

    SELECT *
    FROM names
    JOIN data
        ON data.playerID = names.trash
'''

pitching_data = pd.read_sql(pitching_query, lahmans)
pitching_data.drop(columns='trash', inplace=True)
by_IP = stat.order_by(pitching_data, 'IPouts', False)

In [21]:
by_IP['Innings'] = round(by_IP.IPouts / 3)
pitching_data = by_IP

In [22]:
pitching_2001 = pitching_data[(pitching_data.yearID == 2001) & (pitching_data['Innings'] > 200)]
pitching_2001.head()

,nameFirst,nameLast,ID,playerID,yearID,stint,teamID,team_ID,lgID,W,L,G,GS,CG,SHO,SV,IPouts,H,ER,HR,BB,SO,BAOpp,ERA,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP,Innings
494,Curt,Schilling,34095,schilcu01,2001,1,ARI,2357,NL,22,6,35,35,6,1,0,770,237,85,37,39,293,0.245,2.98,0.0,4,1,0,1021,0,86,8.0,5.0,16.0,257.0
646,Randy,Johnson,33842,johnsra05,2001,1,ARI,2357,NL,21,6,35,34,3,2,0,749,181,69,19,71,372,0.203,2.49,2.0,8,18,1,994,1,74,10.0,5.0,17.0,250.0
882,Freddy,Garcia,33772,garcifr02,2001,1,SEA,2380,AL,18,6,34,34,4,3,0,716,199,81,16,69,163,0.225,3.05,6.0,3,5,1,971,0,88,8.0,5.0,23.0,239.0
983,Tim,Hudson,33826,hudsoti01,2001,1,OAK,2376,AL,18,9,35,35,3,0,0,705,216,88,20,71,181,0.245,3.37,5.0,9,6,1,980,0,100,12.0,8.0,18.0,235.0
1005,Chan Ho,Park,33999,parkch01,2001,1,LAN,2370,NL,15,11,36,35,2,1,0,702,183,91,23,91,218,0.216,3.50,1.0,3,20,3,981,0,98,16.0,7.0,11.0,234.0


In [30]:
era_plus = lambda row: (100 - (100 * row.ERA / lgERA)) + 100

In [32]:
pitching_2001.sort_values(by='ERA+', inplace=True, ascending=False)
pitching_2001.reset_index(drop=True, inplace=True)
top_2s = pitching_2001.head(2)
top_2s

,nameFirst,nameLast,ID,playerID,yearID,stint,teamID,team_ID,lgID,W,L,G,GS,CG,SHO,SV,IPouts,H,ER,HR,BB,SO,BAOpp,ERA,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP,Innings,ERA+
0,Randy,Johnson,33842,johnsra05,2001,1,ARI,2357,NL,21,6,35,34,3,2,0,749,181,69,19,71,372,0.203,2.49,2.0,8,18,1,994,1,74,10.0,5.0,17.0,250.0,126.226270
1,Curt,Schilling,34095,schilcu01,2001,1,ARI,2357,NL,22,6,35,35,6,1,0,770,237,85,37,39,293,0.245,2.98,0.0,4,1,0,1021,0,86,8.0,5.0,16.0,257.0,111.708548


In [40]:
min_year = pitching_data.yearID.min()
max_year = pitching_data.yearID.max()
for i in range(min_year, max_year + 1):
    era_plus = lambda row: (100 - (100 * row.ERA / lgERA)) + 100
    df = pitching_data[(pitching_data.yearID == i) & (pitching_data.yearID >= 200)]
    lgERA = df.yearID.mean()
    df['ERA+'] = df.apply(era_plus, axis=1)
    df.sort_values(by='ERA+', ascending=False, inplace=True)
    df = df.head(2)
    top_2s = pd.concat([top_2s, df])


In [42]:
top_2s.sort_values(by='yearID', inplace=True, ascending=False)
print(len(top_2s[top_2s.yearID == 2019]))

32
